# Rank vs intensity, per marker: kNN enrichment and cell-type purity of the same cells (Danenberg + Jackson IMC)

Follow-up to `KnnEnrichmentIMC` / `KnnEnrichmentJackson` and `ProfileVsPhysical`. Those runs showed, at the panel level, that rank gives higher kNN enrichment of marker-high cells but a lower same-phenotype share among profile neighbours. Here both are measured **per marker, on the same cells and the same neighbours**:

- cells high in marker M (robust z > 1, as in the CyTOF code);
- their 30 nearest profile neighbours in the full panel (36 Danenberg / 32 Jackson), within the image, Euclidean;
- **E**: kNN enrichment = (fraction of neighbours that are also M-high) / (share of M-high cells in the image);
- **same_type**: fraction of neighbours with the cell's own phenotype (Danenberg 32 phenotypes; Jackson 27 metaclusters);
- **same_broad**: the same for broad class (epithelial / immune / stromal). `same_broad − same_type` = neighbours of another phenotype in the same broad class;
- **chance_type**: same-phenotype share of a random other cell in the image (identical for both encodings).

Encodings, filters and image subsampling (max 6000 cells) are those of the earlier runs; touching cells are not excluded here (they were excluded only for the physical-context metrics).
Values:
- `*_cells`: mean over all marker-high cells (pooled over images), with `*_sem_cells` = SEM across those cells.
- `E_intensity`, `E_rank`, etc.: mean over images per patient, then mean over patients, with `*_sem_patients`. These are plotted as bars, with whiskers = 95% CI across patients (mean ± 1.96 × `*_sem_patients`); the paired rank − intensity difference is plotted with its patient-bootstrap CI.
- rank − intensity `_lo`/`_hi`: patient-bootstrap 95% CI. These set the "rank higher / lower in N markers" counts, because cells within a patient are not independent.
Caveat: the phenotypes were defined by clustering intensity data, which favours intensity on purity.

In [ ]:
import os, glob, json, time
import numpy as np, pandas as pd
import pyarrow.parquet as pq
from scipy.spatial.distance import cdist
from joblib import Parallel, delayed
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

SEED = 20260929
OUT = "/kaggle/working"
COFACTOR = 1.0
K = 30
HIGH_Z = 1.0
MIN_HIGH = 10                  # a marker needs >= 10 high cells in an image to be scored there
MIN_CELLS, MAX_CELLS = 200, 6000
N_BOOT = 1000
N_JOBS = os.cpu_count()
STROMAL_KEYS = ("fibro", "endothel", "stroma", "pdpn", "myofib", "vessel", "vwf")   # as ProfileVsPhysical
BROAD_NAMES = ["epithelial", "immune", "stromal"]

def locate():
    """Pair each cells.parquet with its prep_report.json; tell the cohorts apart by their patient column."""
    reps = sorted(glob.glob("/kaggle/input/**/prep_report.json", recursive=True))
    out = {}
    for p in sorted(glob.glob("/kaggle/input/**/cells.parquet", recursive=True)):
        key = "danenberg" if "metabric_id" in pq.read_schema(p).names else "jackson"
        assert key not in out, (key, p, out)
        rep = max(reps, key=lambda r: len(os.path.commonpath([r, p])))
        out[key] = (p, rep)
    print(json.dumps(out, indent=1))
    assert set(out) == {"danenberg", "jackson"}
    return out

## Functions and sanity checks

In [ ]:
def rank_vec(Zs):
    """0 = highest; ties by column order (stable sort), as in the CyTOF code."""
    order = np.argsort(-Zs, axis=1, kind="stable")
    R = np.empty_like(order)
    R[np.arange(len(Zs))[:, None], order] = np.arange(Zs.shape[1])
    return R.astype(np.float32)

def profile_knn(V, k, rng):
    D = cdist(V, V).astype(np.float32)
    D += rng.uniform(0, 1e-5, D.shape).astype(np.float32)     # random tie-breaking
    np.fill_diagonal(D, np.inf)
    return np.argpartition(D, k, axis=1)[:, :k]

def per_image(i, seed, ph_i, br_i, Z_i):
    rng = np.random.default_rng(seed)
    n = len(ph_i)
    if n > MAX_CELLS:
        keep = np.sort(rng.choice(n, MAX_CELLS, replace=False))
        ph_i, br_i, Z_i = ph_i[keep], br_i[keep], Z_i[keep]
        n = MAX_CELLS
    H = Z_i > HIGH_Z
    scored = np.flatnonzero((H.sum(0) >= MIN_HIGH) & (H.mean(0) < 0.9))
    cnt, cnt_b = np.bincount(ph_i), np.bincount(br_i, minlength=3)
    chance = (cnt[ph_i] - 1) / (n - 1)
    chance_b = (cnt_b[br_i] - 1) / (n - 1)
    rows = []
    for enc in ("intensity", "rank"):
        P = profile_knn(Z_i if enc == "intensity" else rank_vec(Z_i), K, rng)
        same = (ph_i[P] == ph_i[:, None]).mean(1)
        same_b = (br_i[P] == br_i[:, None]).mean(1)
        rows.append(dict(img=i, enc=enc, marker=-1, E=np.nan, same_type=same.mean(), same_broad=same_b.mean(),
                         chance_type=chance.mean(), chance_broad=chance_b.mean(), n_high=n,
                         E_sum=np.nan, E_ss=np.nan, T_sum=same.sum(), T_ss=(same ** 2).sum(), C_sum=chance.sum()))
        for m in scored:
            h = H[:, m]
            e_c = h[P[h]].mean(1) / h.mean()          # per-cell enrichment; its mean is the CyTOF score
            t_c = same[h]
            rows.append(dict(img=i, enc=enc, marker=int(m), E=e_c.mean(),
                             same_type=t_c.mean(), same_broad=same_b[h].mean(),
                             chance_type=chance[h].mean(), chance_broad=chance_b[h].mean(), n_high=int(h.sum()),
                             E_sum=e_c.sum(), E_ss=(e_c ** 2).sum(), T_sum=t_c.sum(), T_ss=(t_c ** 2).sum(),
                             C_sum=chance[h].sum()))
    return rows

# sanity: purity of a perfectly separated 2-type profile is 1; enrichment of a marker in the space is > 2
r = np.random.default_rng(0)
Zt = np.r_[r.normal(size=(300, 4)), r.normal(size=(300, 4)) + 8].astype(np.float32)
pt = np.r_[np.zeros(300, int), np.ones(300, int)]
P = profile_knn(Zt, K, r)
assert (pt[P] == pt[:, None]).mean() == 1.0
Zr = r.normal(size=(1500, 6)).astype(np.float32); h = Zr[:, 0] > 1
assert h[profile_knn(Zr, K, r)[h]].mean() / h.mean() > 2
print("sanity checks passed")

## Load both cohorts

In [ ]:
def load(key, paths):
    p, rep = paths[key]
    report = json.load(open(rep))
    cells = pd.read_parquet(p)
    if key == "danenberg":
        chan = list(report["channel_map"].values())
        cells = cells[(cells.is_tumour == 1) & (cells.is_hot_aggregate == 0)]
        pid, n_prot = "metabric_id", 36
    else:
        assert all(report["checks"].values())
        chan = list(report["channel_map"])
        cells = cells[cells.is_tumour == 1]
        pid, n_prot = "patient_id", 32
    protein = [c for c in chan if c not in ("histone_h3", "dna1", "dna2")]
    assert len(protein) == n_prot
    n_per = cells.groupby("image_name").size()
    cells = cells[cells.image_name.isin(n_per.index[n_per >= MIN_CELLS])]
    cells = cells.sort_values(["image_name", "object_number"]).reset_index(drop=True)
    if key == "danenberg":
        epi_frac = cells.groupby("cell_phenotype").is_epithelial.mean()
        broad = {p: "epithelial" if epi_frac[p] > 0.5 else
                 "stromal" if any(s in p.lower() for s in STROMAL_KEYS) else "immune" for p in epi_frac.index}
    else:
        broad = cells.groupby("cell_phenotype").broad_class.first().to_dict()
    print(pd.Series(broad).sort_values().to_string())
    ph_code, ph_names = pd.factorize(cells.cell_phenotype, sort=True)
    br = cells.cell_phenotype.map(broad).map({b: i for i, b in enumerate(BROAD_NAMES)})
    assert br.notna().all()
    img_code, img_names = pd.factorize(cells.image_name, sort=True)
    img_patient = cells.groupby("image_name")[pid].first().reindex(img_names).values
    starts = np.r_[0, np.flatnonzero(np.diff(img_code)) + 1, len(cells)]
    A = np.arcsinh(cells[protein].values.astype(np.float64) / COFACTOR)
    Z = np.empty_like(A, dtype=np.float32)
    for a, b in zip(starts[:-1], starts[1:]):
        med = np.median(A[a:b], 0)
        mad = np.median(np.abs(A[a:b] - med), 0)
        Z[a:b] = np.clip((A[a:b] - med) / (1.4826 * mad + 1e-6), -5, 5)
    print(f"{key}: {len(cells):,} cells, {len(img_names)} images, {len(np.unique(img_patient))} patients, "
          f"{len(ph_names)} phenotypes, {len(protein)} markers")
    return dict(protein=protein, ph=ph_code, br=br.values.astype(int), Z=Z, starts=starts, img_patient=img_patient)

PATHS = locate()

## Run all images and summarise per marker

In [ ]:
VALUES = ["E", "same_type", "same_broad", "chance_type", "chance_broad"]
DIFFS = ["E", "same_type", "same_broad"]

def summarise(d, cohort):
    pp = d.groupby(["marker_name", "enc", "patient"])[VALUES].mean().unstack("enc")
    pooled = d.groupby(["marker_name", "enc"])[["E_sum", "E_ss", "T_sum", "T_ss", "C_sum", "n_high"]].sum(min_count=1)
    rb = np.random.default_rng(SEED)
    rows = []
    for mk, w in pp.groupby(level="marker_name"):
        w = w.dropna(subset=[("same_type", "intensity"), ("same_type", "rank")])
        row = dict(cohort=cohort, marker=mk, n_patients=len(w), n_images=int(d[d.marker_name == mk].img.nunique()))
        # cell level: mean over all marker-high cells (pooled over images) and SEM across those cells
        for enc in ("intensity", "rank"):
            s = pooled.loc[(mk, enc)]
            N = s.n_high
            row["n_cells"] = int(N)
            for v, pre in (("E", "E"), ("same_type", "T")):
                mu = s[f"{pre}_sum"] / N
                var = max(s[f"{pre}_ss"] / N - mu ** 2, 0.0) * N / (N - 1)
                row[f"{v}_{enc}_cells"], row[f"{v}_{enc}_sem_cells"] = mu, np.sqrt(var / N)
            row["chance_type_cells"] = s.C_sum / N
        for v in ("E", "same_type"):
            row[f"{v}_rank_minus_int_cells"] = row[f"{v}_rank_cells"] - row[f"{v}_intensity_cells"]
            for enc in ("intensity", "rank"):         # patient level, for comparison
                x = w[(v, enc)].dropna().values
                row[f"{v}_{enc}_sem_patients"] = x.std(ddof=1) / np.sqrt(len(x)) if len(x) > 1 else np.nan
                row[f"{v}_{enc}_sd_patients"] = x.std(ddof=1) if len(x) > 1 else np.nan
        B = rb.integers(0, len(w), (N_BOOT, len(w)))
        for v in VALUES:
            a, b = w[(v, "intensity")].values, w[(v, "rank")].values
            if v.startswith("chance"):
                row[v] = a.mean()            # identical for both encodings
                continue
            if np.isnan(a).all():
                continue
            row[f"{v}_intensity"], row[f"{v}_rank"] = a.mean(), b.mean()
            dd = b - a
            bs = dd[B].mean(1)
            row[f"{v}_rank_minus_int"] = dd.mean()
            row[f"{v}_rank_minus_int_lo"], row[f"{v}_rank_minus_int_hi"] = np.percentile(bs, [2.5, 97.5])
        # rank's purity loss split: to another phenotype of the same broad class vs to another broad class
        loss = -row["same_type_rank_minus_int"]
        row["type_loss_to_other_broad_class"] = -row["same_broad_rank_minus_int"]
        row["type_loss_within_broad_class"] = loss - row["type_loss_to_other_broad_class"]
        rows.append(row)
    return pd.DataFrame(rows)

per_marker, per_image_all = [], []
for key in ("danenberg", "jackson"):
    D = load(key, PATHS)
    t0 = time.time()
    res = Parallel(n_jobs=N_JOBS, verbose=2)(
        delayed(per_image)(i, SEED + i, D["ph"][a:b], D["br"][a:b], D["Z"][a:b])
        for i, (a, b) in enumerate(zip(D["starts"][:-1], D["starts"][1:])))
    print(f"{key} per-image: {(time.time() - t0) / 60:.1f} min")
    d = pd.DataFrame([r for rows in res for r in rows])
    d["cohort"] = key
    d["patient"] = D["img_patient"][d.img.values]
    d["marker_name"] = np.array(D["protein"] + ["all_cells"])[d.marker.values]     # -1 -> all_cells
    per_image_all.append(d)
    per_marker.append(summarise(d, key))
    del D

per_image_all = pd.concat(per_image_all, ignore_index=True)
per_image_all.to_parquet(f"{OUT}/tradeoff_per_image.parquet", index=False)
pm = pd.concat(per_marker, ignore_index=True)
pm.to_csv(f"{OUT}/tradeoff_per_marker.csv", index=False)
cols = ["cohort", "marker", "E_intensity", "E_rank", "E_rank_minus_int", "same_type_intensity", "same_type_rank",
        "same_type_rank_minus_int", "chance_type", "type_loss_within_broad_class", "type_loss_to_other_broad_class"]
print(pm[cols].round(3).to_string(index=False))

## Summary

In [ ]:
summ = {}
for key, g in pm[pm.marker != "all_cells"].groupby("cohort"):
    a = pm[(pm.cohort == key) & (pm.marker == "all_cells")].iloc[0]
    summ[key] = dict(
        n_markers=len(g),
        enrichment_rank_higher_ci=int((g.E_rank_minus_int_lo > 0).sum()),
        enrichment_rank_lower_ci=int((g.E_rank_minus_int_hi < 0).sum()),
        purity_rank_lower_ci=int((g.same_type_rank_minus_int_hi < 0).sum()),
        purity_rank_higher_ci=int((g.same_type_rank_minus_int_lo > 0).sum()),
        median_E_diff=float(g.E_rank_minus_int.median()),
        median_purity_diff=float(g.same_type_rank_minus_int.median()),
        spearman_Ediff_vs_puritydiff=float(g[["E_rank_minus_int", "same_type_rank_minus_int"]].corr("spearman").iloc[0, 1]),
        all_cells_purity=[float(a.same_type_intensity), float(a.same_type_rank), float(a.chance_type)],
        all_cells_loss_within_broad=float(a.type_loss_within_broad_class),
        all_cells_loss_other_broad=float(a.type_loss_to_other_broad_class))
json.dump(summ, open(f"{OUT}/summary_tradeoff.json", "w"), indent=1)
print(json.dumps(summ, indent=1))

## Figure

In [ ]:
INT_C, RANK_C = "#2a78d6", "#eb6834"          # categorical slots 1-2 (intensity, rank)
INK, INK2, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
LABELS = {
    "pan_cytokeratin": "panCK", "cytokeratin_5": "CK5", "cytokeratin_7": "CK7", "cytokeratin_8_18": "CK8/18",
    "cytokeratin_19": "CK19", "keratin_14_krt14": "CK14", "sma": "SMA", "estrogen_receptor_alpha": "ER",
    "progesterone_receptor_a_b": "PR", "ki_67": "Ki-67", "c_erb_b_2_her2_3b5": "HER2 (3B5)",
    "c_erb_b_2_her2_d8f12": "HER2 (D8F12)", "c_erb_b_2_her2": "HER2", "hla_dr": "HLA-DR", "hla_abc": "HLA-ABC",
    "beta_2_microglobulin": "B2M", "cd278_icos": "ICOS", "cd134": "OX40", "cd279_pd_1": "PD-1",
    "gitr_tnfrsf18": "GITR", "cd140b_pdgf_receptor_beta": "PDGFRB", "caveolin_1": "Caveolin-1",
    "cd31_v_wf": "CD31/vWF", "v_wf": "vWF", "cxcl12_sdf_1": "CXCL12", "cleaved_caspase3": "Cleaved caspase-3",
    "cleaved_parp": "Cleaved PARP", "fsp1": "FSP1", "podoplanin": "Podoplanin", "foxp3": "FOXP3",
    "cd45ra": "CD45RA", "cd8a": "CD8a", "e_cadherin_p_cadherin": "E/P-cadherin", "egfr": "EGFR",
    "gata3": "GATA3", "histone_h3_phospho": "pHH3", "histone_h3_trimethylate": "H3K27me3", "m_tor": "mTOR",
    "s6": "S6", "c_myc": "c-Myc", "p53": "p53", "slug": "Slug", "twist": "Twist", "vimentin": "Vimentin",
    "fibronectin": "Fibronectin", "carbonic_anhydrase_ix": "CA9",
}

def pretty(m):
    return LABELS.get(m, m.upper() if m.startswith("cd") else m.replace("_", " "))

def plot_tradeoff(pm, cohort, path):
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.lines import Line2D
    g = pm[pm.marker != "all_cells"].sort_values("E_rank_minus_int").reset_index(drop=True)
    y, n = np.arange(len(g)), len(g)
    plt.rcParams.update({"font.family": "sans-serif", "font.size": 8.5})
    W, H = 11.0, 0.22 * n + 1.9
    fig, axes = plt.subplots(1, 2, figsize=(W, H), sharey=True)
    fig.patch.set_facecolor("white")
    panels = [("E", "A   kNN enrichment of marker-high cells", "enrichment among the 30 profile neighbours (1 = none)"),
              ("same_type", "B   Same-phenotype share among the same neighbours", "share of the 30 profile neighbours with the same phenotype")]
    for ax, (v, ttl, xl) in zip(axes, panels):
        a, b = g[f"{v}_intensity"].values, g[f"{v}_rank"].values
        ax.hlines(y, np.minimum(a, b), np.maximum(a, b), color=AXIS, lw=2, zorder=1)
        if v == "same_type":
            ax.scatter(g.chance_type, y, marker="|", s=70, color=MUTED, lw=1.5, zorder=2)
        else:
            ax.axvline(1, color=MUTED, lw=1, ls="--", zorder=0)
        for x, c, enc in ((a, INT_C, "intensity"), (b, RANK_C, "rank")):
            ax.errorbar(x, y, xerr=g[f"{v}_{enc}_sem_patients"].values, fmt="none", ecolor=c, elinewidth=1.2,
                        capsize=2.5, capthick=1.2, zorder=4)
            ax.scatter(x, y, s=34, color=c, edgecolor="white", lw=0.8, zorder=3)
        lo, hi = g[f"{v}_rank_minus_int_lo"], g[f"{v}_rank_minus_int_hi"]
        med = np.nanmedian(g[f"{v}_rank_minus_int"])
        ax.annotate(ttl, xy=(0, 1), xycoords="axes fraction", xytext=(0, 27), textcoords="offset points",
                    color=INK, fontsize=10, fontweight="bold", va="bottom")
        ax.annotate(f"{cohort}: rank higher in {int((lo > 0).sum())}, lower in {int((hi < 0).sum())} of {n} markers\n"
                    f"median over the {n} markers of (rank − intensity): {med:+.3f}",
                    xy=(0, 1), xycoords="axes fraction", xytext=(0, 5), textcoords="offset points",
                    color=INK2, fontsize=8, va="bottom")
        ax.set_xlabel(xl, color=INK2)
        for sp in ("top", "right"):
            ax.spines[sp].set_visible(False)
        for sp in ("left", "bottom"):
            ax.spines[sp].set_color(AXIS)
        ax.tick_params(colors=INK2, length=3, labelleft=True)
        ax.set_yticks(y); ax.set_yticklabels([pretty(m) for m in g.marker], color=INK)
    axes[0].set_ylim(-0.7, n - 0.3)
    handles = [Line2D([], [], marker="o", ls="", color=INT_C, ms=6, label="intensity"),
               Line2D([], [], marker="o", ls="", color=RANK_C, ms=6, label="rank"),
               Line2D([], [], marker="|", ls="", color=MUTED, ms=9, mew=1.5, label="chance (B only: random cell, same image)")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.005, 1 - 0.05 / H), ncol=3, frameon=False,
               fontsize=8.5, handletextpad=0.3, columnspacing=1.8)
    fig.text(0.01, 0.06 / H, "Dots: mean over patients; bars: SEM across patients. Markers sorted by rank − intensity enrichment (largest gain at top). "
             "Higher/lower counts markers "
             "whose patient-bootstrap 95% CI excludes 0.", color=MUTED, fontsize=7.5)
    fig.subplots_adjust(left=1.2 / W, right=1 - 0.15 / W, wspace=0.32, top=1 - 1.05 / H, bottom=0.72 / H)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


COHORT_STYLE = {"danenberg": ("Danenberg 2022", "#a6d49f", "#1b7837"),      # light = intensity, dark = rank
                "jackson": ("Jackson 2020", "#9ecae1", "#08519c")}

def plot_tradeoff_combined(pm, path):
    """Rows = cohorts, columns = (kNN enrichment, same-phenotype share). Bars = mean over patients, whiskers = SD."""
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    rows = [c for c in ("danenberg", "jackson") if c in set(pm.cohort)]
    G = {c: pm[(pm.cohort == c) & (pm.marker != "all_cells")].sort_values("E_rank_minus_int").reset_index(drop=True)
         for c in rows}
    plt.rcParams.update({"font.family": "sans-serif", "font.size": 8.5})
    hs = [0.24 * len(G[c]) for c in rows]
    gap = 1.25                                               # inches between rows (titles + x labels)
    W, H = 12.0, sum(hs) + gap * (len(rows) - 1) + 2.1
    fig = plt.figure(figsize=(W, H))
    fig.patch.set_facecolor("white")
    gs = fig.add_gridspec(len(rows), 2, height_ratios=hs, hspace=gap / np.mean(hs), wspace=0.36,
                          left=1.25 / W, right=1 - 0.15 / W, top=1 - 1.35 / H, bottom=0.75 / H)
    panels = [("E", "kNN enrichment of marker-high cells", "enrichment among the 30 profile neighbours (1 = none)"),
              ("same_type", "Same-phenotype share among the same neighbours", "share of the 30 profile neighbours with the same phenotype")]
    letters = iter("ABCD")
    hb = 0.38
    for r, c in enumerate(rows):
        g = G[c]; n = len(g); y = np.arange(n)
        name, light, dark = COHORT_STYLE[c]
        for j, (v, ttl, xl) in enumerate(panels):
            ax = fig.add_subplot(gs[r, j])
            m_i, m_r = g[f"{v}_intensity"].values, g[f"{v}_rank"].values
            s_i, s_r = g[f"{v}_intensity_sd_patients"].values, g[f"{v}_rank_sd_patients"].values
            ax.barh(y + hb / 2, m_i, height=hb, color=light, zorder=2)
            ax.barh(y - hb / 2, m_r, height=hb, color=dark, zorder=2)
            ends = np.r_[m_i + s_i, m_r + s_r]
            right = max(min(np.nanmax(ends), np.nanquantile(ends, 0.95) * 1.1), np.nanmax(np.r_[m_i, m_r]) * 1.05)
            if v == "same_type":
                right = min(right, 1.0)
            for m, s, yy in ((m_i, s_i, y + hb / 2), (m_r, s_r, y - hb / 2)):
                hi = np.minimum(m + s, right)
                ax.hlines(yy, m - s, hi, color=INK2, lw=0.8, zorder=3)
                cut = m + s > right
                ax.plot(np.full(cut.sum(), right), yy[cut], marker=">", ls="", color=INK2, ms=3.5, clip_on=False, zorder=4)
            if v == "same_type":
                ax.scatter(g.chance_type, y, marker="|", s=110, color=INK, lw=1.3, zorder=5)
            else:
                ax.axvline(1, color=INK, lw=0.9, ls="--", zorder=5)
            ax.set_xlim(0, right * 1.02)
            ax.set_ylim(-0.7, n - 0.3)
            lo, hi_ = g[f"{v}_rank_minus_int_lo"], g[f"{v}_rank_minus_int_hi"]
            med = np.nanmedian(g[f"{v}_rank_minus_int"])
            ax.annotate(f"{next(letters)}   {name}: {ttl}", xy=(0, 1), xycoords="axes fraction", xytext=(0, 27),
                        textcoords="offset points", color=INK, fontsize=10, fontweight="bold", va="bottom")
            ax.annotate(f"rank higher in {int((lo > 0).sum())}, lower in {int((hi_ < 0).sum())} of {n} markers\n"
                        f"median over the {n} markers of (rank − intensity): {med:+.3f}",
                        xy=(0, 1), xycoords="axes fraction", xytext=(0, 5), textcoords="offset points",
                        color=INK2, fontsize=8, va="bottom")
            ax.set_yticks(y); ax.set_yticklabels([pretty(mk) for mk in g.marker], color=INK)
            ax.set_xlabel(xl, color=INK2)
            for sp in ("top", "right"):
                ax.spines[sp].set_visible(False)
            for sp in ("left", "bottom"):
                ax.spines[sp].set_color(AXIS)
            ax.tick_params(colors=INK2, length=3)
    handles = []
    for c in rows:
        name, light, dark = COHORT_STYLE[c]
        handles += [Patch(color=light, label=f"{name}, intensity"), Patch(color=dark, label=f"{name}, rank")]
    handles += [Line2D([], [], marker="|", ls="", color=INK, ms=9, mew=1.3, label="chance (random cell, same image)"),
                Line2D([], [], color=INK, lw=0.9, ls="--", label="no enrichment")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.005, 1 - 0.04 / H), ncol=3, frameon=False,
               fontsize=8.5, handlelength=1.4, columnspacing=1.6)
    fig.text(0.01, 0.08 / H, "Bars: mean over patients; whiskers: SD across patients (▸ = whisker truncated at the axis). "
             "Markers sorted by rank − intensity enrichment (largest gain at top). Higher/lower counts markers whose "
             "patient-bootstrap 95% CI excludes 0.", color=MUTED, fontsize=7.5)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


def plot_tradeoff_union(pm, path):
    """One plot per metric with both cohorts; rows = union of markers (shared first).
    Columns: kNN enrichment | rank - intensity | same-phenotype share | rank - intensity."""
    import numpy as np
    import matplotlib.pyplot as plt
    from matplotlib.patches import Patch
    from matplotlib.lines import Line2D
    x = pm[pm.marker != "all_cells"]
    T = {c: x[x.cohort == c].set_index("marker") for c in ("danenberg", "jackson")}
    D, J = T["danenberg"], T["jackson"]
    shared = sorted(set(D.index) & set(J.index), key=lambda m: -(D.E_rank_minus_int[m] + J.E_rank_minus_int[m]) / 2)
    d_only = sorted(set(D.index) - set(J.index), key=lambda m: -D.E_rank_minus_int[m])
    j_only = sorted(set(J.index) - set(D.index), key=lambda m: -J.E_rank_minus_int[m])
    groups = [("Markers in both panels", shared), ("Danenberg panel only", d_only), ("Jackson panel only", j_only)]
    blocks, labels, headers = [], [], []
    y = 0.0
    for gname, ms in groups:
        y -= 1.0; headers.append((y, gname)); y -= 0.9
        for m in ms:
            top = y
            for c in ("danenberg", "jackson"):
                if m in T[c].index:
                    blocks.append((c, m, y - 0.5, y - 1.5)); y -= 2.0
            labels.append(((top + y) / 2, m)); y -= 0.7
    y_min = y
    plt.rcParams.update({"font.family": "sans-serif", "font.size": 8.5})
    unit = 0.072
    W, H = 14.0, -y_min * unit + 2.55
    fig = plt.figure(figsize=(W, H))
    fig.patch.set_facecolor("white")
    gs = fig.add_gridspec(1, 5, width_ratios=[3.3, 1.35, 1.0, 3.3, 1.35], wspace=0.08,
                          left=1.2 / W, right=1 - 0.15 / W, top=1 - 1.45 / H, bottom=1.05 / H)
    axes = [fig.add_subplot(gs[0, i]) for i in (0, 1, 3, 4)]
    metrics = [("E", "A   kNN enrichment of marker-high cells", "enrichment among the 30 profile neighbours (1 = none)"),
               ("same_type", "B   Same-phenotype share among the same neighbours", "share of the 30 neighbours with the same phenotype")]
    hb = 0.9
    for (v, ttl, xl), ax, axd in zip(metrics, axes[0::2], axes[1::2]):
        for c, m, yi, yr in blocks:
            r = T[c].loc[m]
            name, light, dark = COHORT_STYLE[c]
            for yy, enc, col in ((yi, "intensity", light), (yr, "rank", dark)):
                mu, ci = r[f"{v}_{enc}"], 1.96 * r[f"{v}_{enc}_sem_patients"]
                ax.barh(yy, mu, height=hb, color=col, zorder=2)
                ax.hlines(yy, mu - ci, mu + ci, color=INK, lw=1.0, zorder=3)
            if v == "same_type":
                ax.vlines(r.chance_type, yr - hb / 2, yi + hb / 2, color=INK, lw=1.4, zorder=4)
            yc = (yi + yr) / 2
            axd.hlines(yc, r[f"{v}_rank_minus_int_lo"], r[f"{v}_rank_minus_int_hi"], color=dark, lw=1.4, zorder=3)
            axd.plot(r[f"{v}_rank_minus_int"], yc, "o", color=dark, ms=4.2, mec="white", mew=0.6, zorder=4)
        if v == "E":
            ax.axvline(1, color=INK, lw=0.9, ls="--", zorder=5)
        axd.axvline(0, color=INK, lw=0.9, zorder=1)
        for yh, gname in headers:
            ax.text(0.0, yh, gname, transform=ax.get_yaxis_transform(), color=INK2, fontsize=8.5,
                    fontstyle="italic", fontweight="bold", va="center")
        stats = []
        for c in ("danenberg", "jackson"):
            g = T[c]
            lo, hi = g[f"{v}_rank_minus_int_lo"], g[f"{v}_rank_minus_int_hi"]
            stats.append(f"{COHORT_STYLE[c][0]}: rank higher in {int((lo > 0).sum())}, lower in {int((hi < 0).sum())} "
                         f"of {len(g)} markers; median Δ {np.nanmedian(g[f'{v}_rank_minus_int']):+.3f}")
        ax.annotate(ttl, xy=(0, 1), xycoords="axes fraction", xytext=(0, 29), textcoords="offset points",
                    color=INK, fontsize=10, fontweight="bold", va="bottom")
        ax.annotate("\n".join(stats), xy=(0, 1), xycoords="axes fraction", xytext=(0, 5), textcoords="offset points",
                    color=INK2, fontsize=8, va="bottom")
        ax.set_xlabel(xl, color=INK2)
        axd.set_xlabel("Δ = rank − intensity" + chr(10) + "(95% CI)", color=INK2)
        ax.set_yticks([p for p, _ in labels]); ax.set_yticklabels([pretty(m) for _, m in labels], color=INK)
        axd.set_yticks([p for p, _ in labels]); axd.set_yticklabels([])
        ax.set_xlim(0, None)
        for a in (ax, axd):
            a.set_ylim(y_min + 0.3, 0.2)
            for sp in ("top", "right"):
                a.spines[sp].set_visible(False)
            for sp in ("left", "bottom"):
                a.spines[sp].set_color(AXIS)
            a.tick_params(colors=INK2, length=3)
    handles = []
    for c in ("danenberg", "jackson"):
        name, light, dark = COHORT_STYLE[c]
        handles += [Patch(color=light, label=f"{name}, intensity"), Patch(color=dark, label=f"{name}, rank")]
    handles += [Line2D([], [], color=INK, lw=1.4, label="chance (random cell, same image)"),
                Line2D([], [], color=INK, lw=0.9, ls="--", label="no enrichment")]
    fig.legend(handles=handles, loc="upper left", bbox_to_anchor=(0.005, 1 - 0.04 / H), ncol=3, frameon=False,
               fontsize=8.5, handlelength=1.4, columnspacing=1.6)
    fig.text(0.01, 0.1 / H, "Bars: mean over patients; whiskers: 95% CI across patients (mean ± 1.96 SEM). "
             "Δ: paired rank − intensity per marker, 95% patient-bootstrap CI; 'higher/lower' counts markers whose CI excludes 0.\n"
             "Phenotypes are each cohort's published labels (Danenberg: 32 phenotypes; Jackson: 27 metaclusters), "
             "so absolute same-phenotype shares are not comparable between cohorts.", color=MUTED, fontsize=7.5)
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)


plot_tradeoff_union(pm, f"{OUT}/fig_tradeoff_markers.png")
print("done")